# Ingredient / Component Instance Segmentation (YOLOv8-seg) — Kaggle Training

This notebook trains a YOLOv8 **instance segmentation** model (YOLOv8-seg) on a dataset that already has **segmentation polygons** in YOLO format, and saves the trained weights into **Kaggle Output** so you can download them.

## Dataset (works immediately on Kaggle)
Recommended dataset (already YOLO formatted):
- https://www.kaggle.com/datasets/tgiahuy/foodseg103-yolo2

> FoodSeg103 is “food component categories” (103 classes). It’s not a perfect ingredient vocabulary for every dish, but it’s a clean end-to-end demo for training instance segmentation on Kaggle.

## Kaggle steps
1. Kaggle → **New Notebook**
2. **File → Import Notebook → Upload** → upload this notebook (`07_kaggle_train_ingredient_yolov8_seg.ipynb`)
3. Notebook settings (top-right gear):
   - **Accelerator = GPU T4**
   - **Internet = ON** (needed for `pip install` + downloading `yolov8n-seg.pt`)
4. Right panel → **Input → Add Input** → attach `foodseg103-yolo2` (link above)
5. Run the code cells from top to bottom (default is **30 epochs**)
6. When it finishes: **Save Version**
7. Open the saved version → **Output** → download `best_ingredient_yolov8_seg.pt`

## Outputs
- Training run folder: `/kaggle/working/runs/segment/ingredient_yolov8_seg/`
- Copied weight for download: `/kaggle/working/best_ingredient_yolov8_seg.pt`

In [ ]:
# Step 1 — Install dependencies (run once)
import sys, subprocess

def pip_install(pkg: str):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('ultralytics>=8.2.0')
pip_install('pyyaml')

import ultralytics
print('ultralytics:', ultralytics.__version__)

In [ ]:
# Step 2 — Configure training
import os
from pathlib import Path

# Training knobs (recommended demo config for Kaggle T4)
EPOCHS = 30
IMGSZ = 640
BATCH = -1      # -1 lets Ultralytics auto-pick a safe batch size
MODEL_BASE = 'yolov8n-seg.pt'

# Optional: force a specific Kaggle input folder name if you have multiple datasets attached
# Example after attaching the dataset below: KAGGLE_DATASET_DIRNAME = 'foodseg103-yolo2'
# If you attached only ONE dataset, keep this as None (auto-detect).
KAGGLE_DATASET_DIRNAME = None

KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./')
print('WORK_DIR:', WORK_DIR)

if KAGGLE_INPUT.exists():
    print('Datasets under /kaggle/input:')
    for p in sorted(KAGGLE_INPUT.iterdir()):
        if p.is_dir():
            print(' -', p.name)
else:
    print('Not running on Kaggle (no /kaggle/input).')

In [ ]:
# Step 3 — Find a YOLOv8-seg dataset root + data.yaml
#
# Expected (preferred) YOLO layout:
#   <root>/images/train, <root>/images/val
#   <root>/labels/train, <root>/labels/val
# with labels in YOLO-seg polygon format:
#   <class_id> x1 y1 x2 y2 ... (normalized coords)
#
# If the dataset already includes a data.yaml, we will use it.

import re
import yaml

def _looks_like_yolo_root(root: Path) -> bool:
    return (root/'images'/'train').is_dir() and (root/'labels'/'train').is_dir()

def find_dataset_root() -> Path:
    if not KAGGLE_INPUT.exists():
        raise RuntimeError('This finder is meant for Kaggle. Set paths manually for local runs.')

    candidates = []
    if KAGGLE_DATASET_DIRNAME:
        p = KAGGLE_INPUT / KAGGLE_DATASET_DIRNAME
        if not p.exists():
            raise FileNotFoundError(f'KAGGLE_DATASET_DIRNAME not found: {p}')
        candidates = [p]
    else:
        candidates = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]

    def _iter_subdirs_limited(root: Path, max_depth: int = 3):
        # Only walks directories (not files) up to max_depth.
        root = root.resolve()
        for cur, dirnames, _ in os.walk(root):
            cur_p = Path(cur)
            try:
                depth = len(cur_p.relative_to(root).parts)
            except Exception:
                continue
            if depth > max_depth:
                dirnames[:] = []
                continue
            yield cur_p

    # Search up to 3 levels deep for a YOLO root
    for base in candidates:
        if _looks_like_yolo_root(base):
            return base
        for sub in _iter_subdirs_limited(base, max_depth=3):
            if _looks_like_yolo_root(sub):
                return sub
    raise RuntimeError('Could not find YOLO dataset root with images/train and labels/train under /kaggle/input')

def scan_nc_from_labels(labels_dir: Path, max_files: int = 5000) -> int:
    # Find max class id across label txt files.
    # YOLO-seg label line: <cls> x1 y1 x2 y2 ...
    max_cls = -1
    n = 0
    for p in labels_dir.rglob('*.txt'):
        n += 1
        if n > max_files:
            break
        try:
            for line in p.read_text().strip().splitlines():
                line = line.strip()
                if not line:
                    continue
                m = re.match(r'^(\d+)(\s+|$)', line)
                if m:
                    max_cls = max(max_cls, int(m.group(1)))
        except Exception:
            continue
    if max_cls < 0:
        raise RuntimeError(f'No class ids found under {labels_dir}')
    return max_cls + 1

def try_find_names_file(root: Path):
    # Common patterns: classes.txt, names.txt, label_names.json
    for cand in ['classes.txt', 'names.txt', 'class_names.txt']:
        p = root / cand
        if p.is_file():
            names = [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]
            if names:
                return names
    return None

DATASET_ROOT = find_dataset_root()
print('DATASET_ROOT:', DATASET_ROOT)

# Prefer an existing data.yaml if present
existing_yaml = None
for cand in ['data.yaml', 'dataset.yaml', 'yolo.yaml']:
    p = DATASET_ROOT / cand
    if p.is_file():
        existing_yaml = p
        break

if existing_yaml:
    DATA_YAML = existing_yaml
    print('Using existing YAML:', DATA_YAML)
else:
    # Create a minimal data.yaml into /kaggle/working
    nc = scan_nc_from_labels(DATASET_ROOT/'labels')
    names = try_find_names_file(DATASET_ROOT)
    if names is None or len(names) != nc:
        names = [str(i) for i in range(nc)]
        print('⚠ Could not infer class names; using numeric names 0..nc-1')

    data_obj = {
        'path': str(DATASET_ROOT),
        'train': 'images/train',
        'val': 'images/val' if (DATASET_ROOT/'images'/'val').is_dir() else 'images/train',
        'names': names,
    }
    DATA_YAML = WORK_DIR / 'ingredient_seg_data.yaml'
    DATA_YAML.write_text(yaml.safe_dump(data_obj, sort_keys=False))
    print('Wrote YAML:', DATA_YAML)

# Quick sanity checks
assert (DATASET_ROOT/'images'/'train').is_dir(), 'missing images/train'
assert (DATASET_ROOT/'labels'/'train').is_dir(), 'missing labels/train'
print('✓ Dataset looks like YOLOv8-seg format')

In [ ]:
# Step 4 — Train YOLOv8-seg (instance segmentation)
# This writes outputs into /kaggle/working so Kaggle can export them.

from ultralytics import YOLO

print('Loading base model:', MODEL_BASE)
model = YOLO(MODEL_BASE)

results = model.train(
    data=str(DATA_YAML),
    task='segment',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project=str(WORK_DIR / 'runs'),
    name='ingredient_yolov8_seg',
    exist_ok=True,
    pretrained=True
)

print('✓ Training complete')

In [ ]:
# Step 5 — Export/copy the best weights into a simple filename for download
from pathlib import Path
import shutil

run_dir = WORK_DIR / 'runs' / 'segment' / 'ingredient_yolov8_seg'
best = run_dir / 'weights' / 'best.pt'
last = run_dir / 'weights' / 'last.pt'
print('Run dir:', run_dir)
print('Best exists:', best.exists(), best)
print('Last exists:', last.exists(), last)

out_best = WORK_DIR / 'best_ingredient_yolov8_seg.pt'
if best.exists():
    shutil.copy2(best, out_best)
    print('✓ Copied →', out_best)
else:
    print('⚠ best.pt not found — check the training output above')

# List output files (what Kaggle will show under Output)
for p in sorted(WORK_DIR.glob('*.pt')):
    print('Output weight:', p.name, f'({p.stat().st_size/1e6:.1f} MB)')

## After training: download + use locally

### Download from Kaggle Output
1. Click **Save Version**
2. Open the saved version
3. Right panel → **Output** → download `best_ingredient_yolov8_seg.pt`

### Put it into this repo
Copy the downloaded file into the repo `models/` folder, for example:
- `models/best_ingredient_yolov8_seg.pt`

### Note
If the app is set up to load an ingredient-seg model, it can use these masks to show a component breakdown under the total calories.